## 1. 웹스크래핑 연습문제

### 1-1. Daum 뉴스기사 제목 스크래핑하기 

In [1]:
import requests
import bs4
from bs4 import BeautifulSoup

# 다음 경제 뉴스 URL
url = 'https://news.daum.net/economy'
print(url)

# 요청 헤더 설정 : 브라우저 정보
req_header = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.3'
}

# requests 의 get() 함수 호출하기 
res = requests.get(url, headers=req_header)
print(type(res))
print(res.status_code)

# 응답(response)이 OK 이면
if res.ok:
    # 응답(response)에서 text 추출 전 인코딩 설정
    res.encoding = 'utf-8'
    html = res.text
    
    # BeautifulSoup 객체 생성
    soup = BeautifulSoup(html, 'html.parser')

    # CSS 선택자
    a_tags = soup.select("div.main-content a[href*='v']")
    print(type(a_tags))

    # <a> 태그 리스트 순회하기
    for a_tag in a_tags:
        # 1. <a> 태그 내부에서 제목 요소만 가져오기 (Daum 뉴스 주요 제목 클래스: tit_g, tit_txt, link_txt 등)
        title_element = a_tag.select_one('.tit_g, .tit_txt, .link_txt, strong')
        
        if title_element:
            title = title_element.text.strip()
        else:
            # 제목 전용 태그가 없을 경우 <a> 태그 안의 첫 번째 텍스트 요소만 가져옴
            title = a_tag.find(string=True, recursive=False)
            if title:
                title = title.strip()
            else:
                title = a_tag.text.strip() # 예외 상황 처리

        link = a_tag['href']
        print(link)
        print(title)

# 응답(response)이 Error 이면 status code 출력
else:
    print(f'Error code = {res.status_code}')


https://news.daum.net/economy
<class 'requests.models.Response'>
200
<class 'bs4.element.ResultSet'>
https://v.daum.net/v/20260811194459201
[뉴스분석] 상반기 실적으로 본 산업별 흐름
https://v.daum.net/v/20260811191848574
김성환 "11차 전기본 송전망 재검토…신규 철탑 최대 40% 줄일 것"
https://v.daum.net/v/20260811183037434
영업익 10배 뛴 엔씨 "신작 10종 출시·연 매출 5兆 조기 달성"(종합)
https://v.daum.net/v/20260811181129766
‘불장’에 홀려 리밸런싱 미룬 국민연금···수익도 시장 안정도 모두 놓쳤다
https://v.daum.net/v/20260811175610113
中은 대출금리 인하, 美는 칩 담보대출 허용…AI 돈줄 터주기 총력
https://v.daum.net/v/20260811171937721
[단독]6개 대형증권사, 올해 교육세 '3400억' 폭탄…"사실상 횡재세"
https://v.daum.net/v/20260811171154371
폭염에 짧아진 휴가 동선… 호텔 안으로 들어간 여행
https://v.daum.net/v/20260811170407064
"'오디세이', 이 자리에서 보세요" [엔터로그]
https://v.daum.net/v/20260811170248986
제프 딘 떠난 자리…구글 AI 권력은 누구한테 갔나
https://v.daum.net/v/20260811195509490
‘저가 공세’ 中 때린 트럼프… K 태양광 경쟁력 되찾을 호기
https://v.daum.net/v/20260811195355438
경남 아파트 입주전망지수 두 달째 ‘보합’
https://v.daum.net/v/20260811195044357
연회비 최대 700만원… VVIP카드 시장 경쟁 심화
https://v.daum.net/v/20260

### 1-2. Daum 뉴스기사 제목 스크래핑하기 코드를 섹션별로 처리하는 함수로 구현하기

In [2]:
section_dict = {
    '기후/환경': 'climate',
    '사회': 'society',
    '경제': 'economy',
    '정치': 'politics',
    '국제': 'world',
    '문화': 'culture',
    '생활': 'life',
    'IT/과학': 'tech',
    '인물': 'people'
}

# 함수 선언
def print_news(section_name):
    # 1. section_dict에서 영문 섹션 문자열 가져오기
    section_code = section_dict.get(section_name)
    
    # 딕셔너리에 없는 섹션명이 들어왔을 때 예외 처리
    if not section_code:
        print(f"'{section_name}'은(는) 유효한 섹션명이 아닙니다.")
        return

    # 2. URL 생성 및 헤더 설정
    url = f'https://news.daum.net/{section_code}'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    # 3. 구분선 및 요청 URL 출력
    print(f"======> {url} {section_name} 뉴스 <======")

    # 4. HTTP 요청
    res = requests.get(url, headers=headers)
    
    if res.ok:
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # 5. 기사 목록 가져오기 (div.main-content 내부의 v 링크)
        a_tags = soup.select("div.main-content a[href*='v']")
        
        for a_tag in a_tags:
            # 제목 태그만 선택하여 언론사/시간 정보 제거
            title_element = a_tag.select_one('.tit_g, .tit_txt, .link_txt, strong')
            
            if title_element:
                title = title_element.text.strip()
            else:
                # 제목 전용 태그가 없을 경우 직속 텍스트 추출
                title = a_tag.find(string=True, recursive=False)
                if title:
                    title = title.strip()
                else:
                    continue  # 텍스트가 없는 경우 스킵
            
            # 제목이 비어있지 않은 경우에만 출력
            if title:
                link = a_tag['href']
                print(link)
                print(title)
    else:
        print(f'Error code = {res.status_code}')

# 함수 호출
print_news('경제')
print_news('사회')

======> https://news.daum.net/economy 경제 뉴스 <======
https://v.daum.net/v/20260811194459201
[뉴스분석] 상반기 실적으로 본 산업별 흐름
https://v.daum.net/v/20260811191848574
김성환 "11차 전기본 송전망 재검토…신규 철탑 최대 40% 줄일 것"
https://v.daum.net/v/20260811183037434
영업익 10배 뛴 엔씨 "신작 10종 출시·연 매출 5兆 조기 달성"(종합)
https://v.daum.net/v/20260811181129766
‘불장’에 홀려 리밸런싱 미룬 국민연금···수익도 시장 안정도 모두 놓쳤다
https://v.daum.net/v/20260811175610113
中은 대출금리 인하, 美는 칩 담보대출 허용…AI 돈줄 터주기 총력
https://v.daum.net/v/20260811171937721
[단독]6개 대형증권사, 올해 교육세 '3400억' 폭탄…"사실상 횡재세"
https://v.daum.net/v/20260811171154371
폭염에 짧아진 휴가 동선… 호텔 안으로 들어간 여행
https://v.daum.net/v/20260811170407064
"'오디세이', 이 자리에서 보세요" [엔터로그]
https://v.daum.net/v/20260811170248986
제프 딘 떠난 자리…구글 AI 권력은 누구한테 갔나
https://v.daum.net/v/20260811195509490
‘저가 공세’ 中 때린 트럼프… K 태양광 경쟁력 되찾을 호기
https://v.daum.net/v/20260811195355438
경남 아파트 입주전망지수 두 달째 ‘보합’
https://v.daum.net/v/20260811195044357
연회비 최대 700만원… VVIP카드 시장 경쟁 심화
https://v.daum.net/v/20260811194459201
[뉴스분석] 상반기 실적으로 본 산업별 흐름
https://v.d

## 2. 웹스크래핑 연습문제

### 2-1. Nate 뉴스기사 제목 스크래핑하기 (선택)

In [3]:
from urllib.parse import urljoin
from IPython.display import Image, display

# Nate 뉴스 섹션 딕셔너리
nate_section_dict = {
    '최신뉴스': 'n0100',
    '정치': 'n0200',
    '경제': 'n0300',
    '사회': 'n0400',
    '세계': 'n0500',
    'IT/과학': 'n0600'
}

def print_nate_news(section_name):
    # 1. section_dict에서 mid 값 가져오기
    mid = nate_section_dict.get(section_name)
    if not mid:
        print(f"'{section_name}'은(는) 올바른 섹션명이 아닙니다.")
        return

    # 2. URL 및 요청 헤더 설정
    base_url = "https://news.nate.com/recent"
    url = f"{base_url}?mid={mid}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    print(f"\n======> {url} Nate {section_name} 뉴스 <======")

    # 3. HTTP 요청
    res = requests.get(url, headers=headers)
    
    if res.ok:
        res.encoding = 'euc-kr'  # Nate 뉴스의 한글 인코딩
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # 4. 기사 목록 영역 선택 (Nate 최신뉴스 리스트 구조)
        news_cards = soup.select("div.mlt01")
        
        for card in news_cards:
            # <a> 태그 및 링크 추출
            a_tag = card.select_one("a.lt1")
            if not a_tag:
                continue
            
            link = urljoin("https://news.nate.com", a_tag.get('href', ''))
            
            # 제목 추출
            title_element = card.select_one("h2.tit, strong.tit")
            title = title_element.text.strip() if title_element else a_tag.text.strip()
            
            # 5. 이미지 요소 존재 여부 체크 및 처리
            img_element = card.select_one("span.tb img, img")
            img_url = None
            
            if img_element and img_element.get('src'):
                raw_src = img_element.get('src')
                # urljoin을 사용해 '//thumbnews.nateimg.co.kr/...' 형태의 상대/프로토콜 생략 경로를 완전한 URL로 결합
                img_url = urljoin("https:", raw_src)

            # 6. 결과 출력
            print(f"\n[기사 제목] {title}")
            print(f"[기사 링크] {link}")
            
            if img_url:
                print(f"[이미지 URL] {img_url}")
                # Jupyter Notebook 상에서 이미지 출력
                try:
                    display(Image(url=img_url))
                except Exception as e:
                    print(f"(이미지 로드 실패: {e})")
            else:
                print("[이미지] 없음")
                
            print("-" * 60)
            
    else:
        print(f"Error code = {res.status_code}")

# --- 함수 실행 테스트 ---
print_nate_news('경제')
# print_nate_news('IT/과학')


======> https://news.nate.com/recent?mid=n0300 Nate 경제 뉴스 <======

[기사 제목] [인도증시] 유가 상승에 인플레 우려 커지며 하락…유가 영향 큰 소비재 부문 낙폭 커
[기사 링크] https://news.nate.com/view/20260811n32070?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/ni/2026/08/11/2608111949187310.jpg


------------------------------------------------------------

[기사 제목] 현대차 노조 또 부분 파업…신차 흥행 '찬물'
[기사 링크] https://news.nate.com/view/20260811n32066?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/ob/2026/08/11/1533703_732555_1356.jpg


------------------------------------------------------------

[기사 제목] LG전자 "시제품 해외 보내지 않고 국내 인증"
[기사 링크] https://news.nate.com/view/20260811n32049?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/sg/2026/08/11/20260811519427.jpg


------------------------------------------------------------

[기사 제목] '저가 공세' 中 때린 트럼프…K 태양광 경쟁력 되찾을 호기
[기사 링크] https://news.nate.com/view/20260811n32032?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/sg/2026/08/11/20260811520427.jpg


------------------------------------------------------------

[기사 제목] KT, AI 기능 결합 '초이스 요금제' 출시
[기사 링크] https://news.nate.com/view/20260811n32019?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/sg/2026/08/11/20260811520482.jpg


...

------------------------------------------------------------

[기사 제목] [단독] 민주당 "금융사 CEO 연임 막아라" 요구에도…금융당국 "법적 수단 없어"
[기사 링크] https://news.nate.com/view/20260811n30727?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/aj/2026/08/11/20260811194151545067.jpg


------------------------------------------------------------

[기사 제목] "로봇 덕분에" 달리는 삼성전자…"HBM 때문에" 맥 빠진 하이닉스
[기사 링크] https://news.nate.com/view/20260811n26687?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/na/2026/08/11/7674086_high.jpg


------------------------------------------------------------

[기사 제목] "한국 와서 예뻐졌어요"…외국인, 피부과-성형외과서 카드 긁었다
[기사 링크] https://news.nate.com/view/20260811n27101?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/do/2026/08/11/134459079.1.jpg


------------------------------------------------------------

[기사 제목] 김윤덕 "세제개편 단계적 시행…공급 확대위해 그린벨트 해제도 검토"
[기사 링크] https://news.nate.com/view/20260811n31866?mid=n0300
[이미지 URL] https://thumbnews.nateimg.co.kr/news90///news.nateimg.co.kr/orgImg/ck/2026/08/11/kuk20260811000399.950x.0.jpg


------------------------------------------------------------


### 2-2. 하나의 네이버 웹툰과 1개의 회차에 대한 Image 다운로드 하기 (필수)

In [4]:
import os
import requests
from bs4 import BeautifulSoup

def download_one_episode(title, no, url):
    # 1. 헤더 설정 (네이버 웹툰 이미지 서버는 referer를 체크하므로 필수)
    req_header = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'referer': url
    }
    
    # 2. 웹툰 회차 페이지 요청
    res = requests.get(url, headers=req_header)
    
    if res.ok:
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # 3. 이미지 URL 리스트 추출 (제공된 소스와 같이 'IMAG01'이 포함된 이미지 타겟팅)
        imgurl_list = []
        for img_tag in soup.select("img[src*='IMAG01']"):
            imgurl_list.append(img_tag['src'])
            
        print(f"[{title} {no}화] 총 {len(imgurl_list)}개의 이미지를 찾았습니다.")
        
        # 만약 'IMAG01'로 찾아지지 않는 최신 뷰어 구조일 경우를 위한 범용(예비) 선택자
        if len(imgurl_list) == 0:
            for img_tag in soup.select("div.wt_viewer img"):
                imgurl_list.append(img_tag.get('src', ''))
        
        # 4. 저장할 디렉토리 경로 생성 (img\title\no)
        dir_name = os.path.join('img', title, str(no))
        if not os.path.isdir(dir_name):
            os.makedirs(dir_name) # 상위 디렉토리(img, title)가 없어도 한 번에 생성
            
        # 5. 이미지 다운로드 및 저장
        for idx, img_url in enumerate(imgurl_list, 1):
            img_res = requests.get(img_url, headers=req_header)
            
            if img_res.ok:
                img_data = img_res.content
                file_name = os.path.basename(img_url)
                # 최종 파일 저장 경로 구성
                file_path = os.path.join(dir_name, file_name)
                
                with open(file_path, 'wb') as file:
                    file.write(img_data)
                    print(f"[{idx}/{len(imgurl_list)}] {file_path} (파일크기: {len(img_data)} bytes)")
            else:
                print(f"[{idx}] 이미지 다운로드 실패 (상태 코드: {img_res.status_code})")
    else:
        print(f"페이지 요청 실패 (상태 코드: {res.status_code})")

# 함수 호출
download_one_episode('일렉시드', 341, 'https://comic.naver.com/webtoon/detail?titleId=717481&no=341&week=wed')

[일렉시드 341화] 총 88개의 이미지를 찾았습니다.
[1/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_1.jpg (파일크기: 87143 bytes)
[2/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_2.jpg (파일크기: 256127 bytes)
[3/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_3.jpg (파일크기: 184536 bytes)
[4/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_4.jpg (파일크기: 182867 bytes)
[5/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_5.jpg (파일크기: 112615 bytes)
[6/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_6.jpg (파일크기: 169889 bytes)
[7/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_7.jpg (파일크기: 157876 bytes)
[8/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_8.jpg (파일크기: 181837 bytes)
[9/88] img\일렉시드\341\20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_9.jpg (파일크기: 203632 bytes)
[10/88] img\일렉시드\341\20250311184953_812848a288eb3e6c